# 7.4 Hierarchical Clustering

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 7: Clustering Techniques

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-07-clustering-techniques/07.04_hierarchical-clustering.ipynb)

Companion code for Section 7.4 of the book. Run the cells in order from top to bottom.

**Example 7.4.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

np.random.seed(42)
X, y_true = make_blobs(n_samples=200, centers=4, cluster_std=0.7, random_state=42)
X = StandardScaler().fit_transform(X)

# ── Compare linkage methods ───────────────────────────────────────
print('Linkage Method Comparison:')
print(f'{"Linkage":<12} {"Silhouette":>12} {"Cluster sizes"}')
print('='*55)
for linkage_method in ['ward','complete','average','single']:
    agg = AgglomerativeClustering(n_clusters=4, linkage=linkage_method)
    labels = agg.fit_predict(X)
    sil    = silhouette_score(X, labels)
    sizes  = dict(zip(*np.unique(labels, return_counts=True)))
    print(f'{linkage_method:<12} {sil:>12.4f}  {sizes}')

# ── Dendrograms for each linkage ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Hierarchical Clustering - Linkage Method Comparison',
            fontsize=14, fontweight='bold')

for ax, method in zip(axes.ravel(), ['ward','complete','average','single']):
    Z = linkage(X, method=method)
    dendrogram(Z, ax=ax, truncate_mode='lastp', p=20,
              leaf_rotation=45, leaf_font_size=8,
              color_threshold=0.7*max(Z[:,2]),
              above_threshold_color='gray')
    ax.set_title(f'{method.capitalize()} Linkage', fontsize=12, fontweight='bold')
    ax.set_xlabel('Sample Index (or Cluster Size)')
    ax.set_ylabel('Distance')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('hierarchical_dendrograms.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Choosing cut height from dendrogram ───────────────────────────
Z_ward = linkage(X, method='ward')

print('\nWard linkage merge distances (last 10 merges):')
print('(Large jump in distance = natural cluster boundary)')
print(f'{"Merge":>8} {"Distance":>12} {"Gap from prev":>15}')
print('='*40)
dists  = Z_ward[:, 2]
for i in range(-10, 0):
    gap = dists[i] - dists[i-1] if i > -len(dists) else 0
    print(f'{i+len(dists):>8} {dists[i]:>12.4f} {gap:>15.4f}')

# Cut the dendrogram at k=4
labels_ward = fcluster(Z_ward, t=4, criterion='maxclust')
print(f'\nWard k=4: silhouette={silhouette_score(X, labels_ward):.4f}')
print(f'Cluster sizes: {dict(zip(*np.unique(labels_ward, return_counts=True)))}')

# ── Cluster result visualisation ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Hierarchical Clustering Results', fontsize=14, fontweight='bold')
palette = ['#E53935','#1976D2','#43A047','#FF6F00']

for ax, (method, labels_method) in zip(axes, [
    ('Ward',     labels_ward-1),
    ('Complete', AgglomerativeClustering(n_clusters=4,linkage='complete').fit_predict(X)),
    ('Single',   AgglomerativeClustering(n_clusters=4,linkage='single').fit_predict(X)),
]):
    for k in range(4):
        mask = labels_method == k
        ax.scatter(X[mask,0], X[mask,1], s=20, alpha=0.7, color=palette[k])
    ax.set_title(f'{method} Linkage (k=4)'); ax.set_xlabel('Feature 1')
    sil_val = silhouette_score(X, labels_method)
    ax.set_ylabel(f'Silhouette = {sil_val:.3f}')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('hierarchical_results.png', dpi=150, bbox_inches='tight')
plt.show()